### 구조화된 출력
- 프로그램에서 쓸때는 정해진 형태 필요 -> json
- 강제로 json 형태로 받도록 할 수 있다

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [3]:
import pandas as pd

data = pd.read_csv("../data/11-1_뉴스정제.csv")
data['본문'][1]

'전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 30일 전북 전주시 전주교도소에서 보석으로 석방돼 기자들의 질의에 답변하고 있다. 2022.06.30. pmkeul newsis.com 서울 뉴시스 박정규 기자 이스타항공이 지난 30일 출소한 이상직 전 국회의원에 대해 이스타항공과 전혀 무관한 관계 라고 강조하면서 오해를 야기할 수 있는 언동을 하지 말 것을 경고했다. 이스타항공은 3일 설명자료를 내고 현재까지도 이스타항공이 이 전 의원과 관계 있다고 오해될 여지가 있어 전혀 무관함을 분명히 하고자 한다 며 이같이 밝혔다. 앞서 이 전 의원은 법원의 보석 허가로 전주교도소에서 출소하는 과정에서 취재진들에게 이스타항공이 좋은 회사가 되게끔 하겠다 며 해고된 직원들이 다시 취업하도록 돕겠다는 취지의 발언을 한 바 있다. 이에 대해 이스타항공은 단순히 부적절한 정도를 넘어 새롭게 탈바꿈을 하고 재운항을 준비하고 있는 이스타항공의 진정성 있는 노력에 대내외적 불신을 야기할 수 있는 매우 심각한 문제 라며 향후 이스타항공과 관련이 있는 것으로 오해가 될 수 있는 어떠한 언동도 금해주시기를 요청드린다 고 밝혔다. 또 다시 이러한 일이 발생할 경우 재발방지를 위한 모든 조치를 강구할 것임을 분명히 밝힌다 고 덧붙였다. 이스타항공은 서울회생법원으로부터 인가된 회생계획에 따라 기존 최대주주인 이스타홀딩스 보유주식을 포함한 구주 전체가 소각됐다 면서 이 전 의원 측은 서울회생법원의 회생절차에서 어떠한 관여도 할 수 없었으며 회생계획에 따른 구주 전체의 무상소각 이후 이스타항공의 주식을 단 1주도 보유하고 있지 않은 이스타항공과 전혀 무관한 관계 라고 선을 그었다. 아울러 이스타항공을 인수한 주식회사 성정 또한 이 전 의원과 전혀 관계가 없으며 특히 형남순 회장을 비롯한 관계인 그 누구도 이 전 의원과 일면식조차 없다 고 강조했다.'

In [4]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{
        "role" : "user",
        "content" : f"다음 기사에서 카테고리와 핵심어를 json으로 줘. {data['본문'][1]}"
    }]
)

print(response.choices[0].message.content)

{
  "카테고리": "항공·기업",
  "핵심어": [
    "이스타항공",
    "이상직 전 의원",
    "배임·횡령",
    "보석 석방",
    "전주교도소",
    "이스타항공과 무관",
    "회생절차",
    "구주 소각",
    "성정",
    "형남순 회장",
    "해고 직원 재취업",
    "재발방지 조치"
  ]
}


In [7]:
# 그럼 이렇게 dict로 처리할 수 있다

import json
output = json.loads(response.choices[0].message.content)
type(output)

dict

### 방법 1. 스키마(원하는 형식)로 원하는 형식 강제하기
- response_format에 원하는 형식 지정
- strice = True

In [ ]:
# 복잡하다!
schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "news_info",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "카테고리": {"type": "string"},
                "핵심어": {"type": "array", "items": {"type": "string"}},
                "한줄요약": {"type": "string"},
            },
            "required": ["카테고리", "핵심어", "한줄요약"],
            "additionalProperties": False,
        },
    },
}

In [9]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    response_format=schema,
    messages=[{
        "role" : "user",
        "content" : f"다음 기사에서 카테고리와 핵심어를 json으로 줘. {data['본문'][1]}"
    }]
)

print(response.choices[0].message.content)

{"카테고리":"사회·기업","핵심어":["이스타항공","이상직 전 의원","보석 석방","배임·횡령","전주교도소","이스타항공과 무관","구주 전량 소각","서울회생법원","성정 인수","재운항 준비"],"한줄요약":"이스타항공이 보석으로 석방된 이상직 전 의원과 회사 및 인수자 성정은 전혀 무관하다며 관련 발언을 자제해 달라고 경고했다."}


In [10]:
result_dict = json.loads(response.choices[0].message.content)
result_dict

{'카테고리': '사회·기업',
 '핵심어': ['이스타항공',
  '이상직 전 의원',
  '보석 석방',
  '배임·횡령',
  '전주교도소',
  '이스타항공과 무관',
  '구주 전량 소각',
  '서울회생법원',
  '성정 인수',
  '재운항 준비'],
 '한줄요약': '이스타항공이 보석으로 석방된 이상직 전 의원과 회사 및 인수자 성정은 전혀 무관하다며 관련 발언을 자제해 달라고 경고했다.'}

In [11]:
result_dict['한줄요약']

'이스타항공이 보석으로 석방된 이상직 전 의원과 회사 및 인수자 성정은 전혀 무관하다며 관련 발언을 자제해 달라고 경고했다.'

### 2. Pydantic으로 제어하기
- 모델이 원하는 구조를 파이썬 클래스처럼 정의해서 사용

In [12]:
from pydantic import BaseModel, Field

class NewsInfo(BaseModel):
    카테고리 : str = Field(description="경제, IT과학, 정치 등")
    핵심어 : list[str]
    한줄요약 : str
    뉴스기사완성도 : int = Field(description="1~10점")

In [ ]:
response = client.chat.completions.parse(   # 이때는 create가 아니라 parse로 해야 한다!!
    model="gpt-5.6-luna",
    response_format=NewsInfo,   #이렇게 넣으면 된다!
    messages=[{
        "role" : "user",
        "content" : f"다음 기사를 분석해줘. {data['본문'][1]}"
    }]
)

print(response.choices[0].message.content)

result_dict = json.loads(response.choices[0].message.content)
result_dict

{"카테고리":"경제·사회","핵심어":["이스타항공","이상직 전 의원","배임·횡령","보석 석방","전주교도소","서울회생법원","구주 소각","주식회사 성정"],"한줄요약":"이스타항공은 보석으로 석방된 이상직 전 의원의 발언과 관련해 회사와 이 전 의원은 현재 아무런 관계가 없다며, 오해를 불러일으킬 수 있는 언동을 자제해 달라고 경고했다.","뉴스기사완성도":9}


{'카테고리': '경제·사회',
 '핵심어': ['이스타항공',
  '이상직 전 의원',
  '배임·횡령',
  '보석 석방',
  '전주교도소',
  '서울회생법원',
  '구주 소각',
  '주식회사 성정'],
 '한줄요약': '이스타항공은 보석으로 석방된 이상직 전 의원의 발언과 관련해 회사와 이 전 의원은 현재 아무런 관계가 없다며, 오해를 불러일으킬 수 있는 언동을 자제해 달라고 경고했다.',
 '뉴스기사완성도': 9}

In [17]:
# 정해진 카테고리에서 고를 땐 아래와 같이 사용해야!

from typing import Literal

class NewsInfo(BaseModel):
    카테고리 : Literal[
        "경제", "IT과학", "사회", "항공", "문화", "스포츠", "국제"
    ] = Field(description="본문 내용을 바탕으로 카테고리 설정")
    핵심어 : list[str]
    한줄요약 : str
    뉴스기사완성도 : int = Field(description="1~10점")


In [18]:
response = client.chat.completions.parse(   # 이때는 create가 아니라 parse로 해야 한다!!
    model="gpt-5.6-luna",
    response_format=NewsInfo,   #이렇게 넣으면 된다!
    messages=[{
        "role" : "user",
        "content" : f"다음 기사를 분석해줘. {data['본문'][1]}"
    }]
)

print(response.choices[0].message.content)

{"카테고리":"항공","핵심어":["이스타항공","이상직 전 의원","보석 석방","회생절차","성정"],"한줄요약":"이스타항공이 보석으로 석방된 이상직 전 의원의 발언과 관련해 회사 및 인수자인 성정과 전혀 무관하다며 오해를 유발할 수 있는 언동을 중단하라고 경고했다.","뉴스기사완성도":9}


### 중첩 스키마
- pydantic의 모델 안에 다른 모델

In [ ]:
# 전체 내용을 스키마로 정의
from typing import Optional # 해당하는 항목이 있거나 없기도 할 때

class Person(BaseModel):
    name : str = Field(description="인물 이름")
    role : str = Field(description="기사에서 역할이나 직함")

class ArticleEntity(BaseModel):
    title : str = Field(description="전체 기사내용을 한줄로")
    people : list[Person]
    date : Optional[str] = Field(description="기사 속 사건 날짜. YYYY-MM-DD") # 기사에 날짜 없을 수도 있으므로 optional 사용

# 정리: 클래스 안에서 정의된 요소도 class 요소 갖고 이ㅆ도록

In [25]:
response = client.chat.completions.parse(   # 이때는 create가 아니라 parse로 해야 한다!!
    model="gpt-5.6-luna",
    response_format=ArticleEntity,   #이렇게 넣으면 된다!
    messages=[{
        "role" : "user",
        "content" : f"다음 기사에서 정보를 추출해줘. {data['본문'][1]}"
    }]
)

result_dict = json.loads(response.choices[0].message.content)
result_dict

{'title': '이스타항공, 보석 석방된 이상직 전 의원과 전혀 무관하다며 관련 언동 자제 경고',
 'people': [{'name': '이상직',
   'role': '전 국회의원 및 이스타항공 전 의원; 자금 배임·횡령 혐의로 수감됐다가 보석 석방'},
  {'name': '김얼', 'role': '뉴시스 전주 지역 기자'},
  {'name': '박정규', 'role': '뉴시스 서울 지역 기자'},
  {'name': '형남순', 'role': '주식회사 성정 회장'}],
 'date': '2022-06-30'}